# Google One AI Kit — 02. 专属风格 LoRA 训练（L4 24GB / A100）

> **开始前必须满足 3 个条件**：
> 1. **数据集已就绪**：在本地用 `prep_lora_dataset.py --sync-to-drive` 准备 15~25 张风格锚点图和同名 `.txt` 标注到 Google Drive `GoogleOneAI_Studio/datasets/custom_style_v1/`。
> 2. **已同意 FLUX 协议**：登录 Hugging Face，在 [`black-forest-labs/FLUX.1-dev`](https://huggingface.co/black-forest-labs/FLUX.1-dev) 页面点击同意开源访问条款。
> 3. **已配 Colab Secret**：点击 Colab 左侧栏 🔑 **Secrets** -> 添加 `HF_TOKEN`（Read 权限 Token）并打开 **Notebook access** 开关。
>
> **硬件与耗时参考**：L4 (24GB) 约 2.5~3.5 小时（消耗约 4~6 计算单元）；A100 (40GB/80GB) 约 40~70 分钟（消耗约 8~12 计算单元）。

In [ ]:
# @title 1. 检查 GPU/VRAM、挂载 Google Drive 并验证 HF_TOKEN 权限
import os
import subprocess
import sys
import torch
from google.colab import drive, userdata

subprocess.run(["nvidia-smi"], check=False)

if not torch.cuda.is_available():
    raise RuntimeError("❌ 未检测到 GPU！请选择 L4 GPU 或 A100 GPU。")

VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"\n✅ 当前 GPU: {torch.cuda.get_device_name(0)} | 显存: {VRAM_GB:.1f} GB")
if VRAM_GB < 22.0:
    raise RuntimeError("❌ FLUX LoRA 训练至少需要 24 GB 显存（L4 或 A100），T4 (16GB) 不足以稳定训练。")

drive.mount("/content/drive")

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as exc:
    raise RuntimeError(
        "❌ 未读取到 Colab Secret 'HF_TOKEN'！\n"
        "请点击左侧栏 🔑 Secrets -> 添加 HF_TOKEN，并开启 Notebook access 权限。"
    ) from exc

if not HF_TOKEN or not HF_TOKEN.strip():
    raise RuntimeError("❌ HF_TOKEN 为空，请检查 Colab Secrets。")

os.environ["HF_TOKEN"] = HF_TOKEN.strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub"], check=True)

from huggingface_hub import HfApi
try:
    info = HfApi(token=os.environ["HF_TOKEN"]).model_info("black-forest-labs/FLUX.1-dev")
    print(f"✅ Hugging Face 权限校验通过: {info.id}")
except Exception as exc:
    raise RuntimeError(
        "❌ 无法访问 black-forest-labs/FLUX.1-dev！\n"
        "请确认：1) 已在浏览器打开 https://huggingface.co/black-forest-labs/FLUX.1-dev 并点击 Agree 同意条款；"
        "2) HF_TOKEN 有效且拥有 Read 权限。"
    ) from exc

In [ ]:
# @title 2. 检查训练集图片与同名 .txt 标注是否完整
import glob
import os

DATASET_NAME = "custom_style_v1"  # @param {type:"string"}
DATASET_DIR = f"/content/drive/MyDrive/GoogleOneAI_Studio/datasets/{DATASET_NAME}"

if not os.path.isdir(DATASET_DIR):
    raise FileNotFoundError(f"❌ 找不到数据集目录: {DATASET_DIR}")

exts = (".png", ".jpg", ".jpeg")
images = sorted(
    p for p in glob.glob(os.path.join(DATASET_DIR, "*"))
    if os.path.splitext(p)[1].lower() in exts
)

missing_txt = []
for img in images:
    txt = os.path.splitext(img)[0] + ".txt"
    if not os.path.exists(txt):
        missing_txt.append(os.path.basename(img))

print(f"📂 数据集目录: {DATASET_DIR}")
print(f"🖼️ 有效图片数量: {len(images)} 张")

if missing_txt:
    raise FileNotFoundError(
        f"❌ 以下图片缺少同名 .txt 标注文件（请先在本地跑 prep_lora_dataset.py）: {missing_txt}"
    )

if len(images) < 5:
    raise ValueError(f"❌ 训练集仅有 {len(images)} 张图片，至少需要 5 张（推荐 15~25 张）才能训练。")
if len(images) < 15:
    print(f"⚠️ 当前仅 {len(images)} 张图片，建议凑满 15~25 张以获得更稳定的风格泛化能力。")
else:
    print("✅ 数据集质量与配对检查通过！")

In [ ]:
# @title 3. 安装 ostris/ai-toolkit 训练框架
import os
import subprocess
import sys

TOOLKIT_DIR = "/content/ai-toolkit"
if not os.path.exists(TOOLKIT_DIR):
    subprocess.run(["git", "clone", "https://github.com/ostris/ai-toolkit.git", TOOLKIT_DIR], check=True)
    subprocess.run(["git", "submodule", "update", "--init", "--recursive"], cwd=TOOLKIT_DIR, check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{TOOLKIT_DIR}/requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "bitsandbytes"], check=True)
print("✅ ai-toolkit 安装完成")

In [ ]:
# @title 4. 生成训练配置文件（自动按 L4 24GB / A100 显存优化）
import os
import torch
import yaml

RUN_NAME = "custom_style_v1"  # @param {type:"string"}
DATASET_NAME = "custom_style_v1"  # @param {type:"string"}
TRIGGER_WORD = "mystyle"  # @param {type:"string"}
STEPS = 1200  # @param {type:"integer"}
RANK = 16  # @param {type:"integer"}
LR = 4e-4  # @param {type:"number"}
SAVE_EVERY = 250  # @param {type:"integer"}

STUDIO_DIR = "/content/drive/MyDrive/GoogleOneAI_Studio"
DATASET_DIR = f"{STUDIO_DIR}/datasets/{DATASET_NAME}"
TRAINING_DIR = f"{STUDIO_DIR}/training_runs"
os.makedirs(TRAINING_DIR, exist_ok=True)

vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
is_24gb_class = vram_gb < 36.0
print(f"检测显存 {vram_gb:.1f} GB -> {'启用 24GB L4 低显存量化模式 (quantize + low_vram)' if is_24gb_class else '启用 A100 高显存高速模式'}")

cfg = {
    "job": "extension",
    "config": {
        "name": RUN_NAME,
        "process": [
            {
                "type": "sd_trainer",
                "training_folder": TRAINING_DIR,
                "device": "cuda:0",
                "trigger_word": TRIGGER_WORD,
                "network": {
                    "type": "lora",
                    "linear": int(RANK),
                    "linear_alpha": int(RANK),
                },
                "save": {
                    "dtype": "float16",
                    "save_every": int(SAVE_EVERY),
                    "max_step_saves_to_keep": 4,
                    "push_to_hub": False,
                },
                "datasets": [
                    {
                        "folder_path": DATASET_DIR,
                        "caption_ext": "txt",
                        "caption_dropout_rate": 0.05,
                        "shuffle_tokens": False,
                        "cache_latents_to_disk": True,
                        "resolution": [512, 768, 1024],
                    }
                ],
                "train": {
                    "batch_size": 1,
                    "steps": int(STEPS),
                    "gradient_accumulation_steps": 1,
                    "train_unet": True,
                    "train_text_encoder": False,
                    "gradient_checkpointing": True,
                    "noise_scheduler": "flowmatch",
                    "optimizer": "adamw8bit",
                    "lr": float(LR),
                    "ema_config": {
                        "use_ema": True,
                        "ema_decay": 0.99,
                    },
                    "dtype": "bf16",
                },
                "model": {
                    "name_or_path": "black-forest-labs/FLUX.1-dev",
                    "is_flux": True,
                    "quantize": bool(is_24gb_class),
                    "low_vram": bool(is_24gb_class),
                },
                "sample": {
                    "sampler": "flowmatch",
                    "sample_every": int(SAVE_EVERY),
                    "sample_start_step": 0,
                    "width": 768,
                    "height": 1152,
                    "prompts": [
                        "[trigger], full body stylized concept art of a futuristic explorer standing in a clean dark studio, crisp rim lighting",
                        "[trigger], wide atmospheric environment backdrop at night with dramatic volumetric lighting and clean foreground",
                    ],
                    "neg": "",
                    "seed": 42,
                    "walk_seed": True,
                    "guidance_scale": 3.5,
                    "sample_steps": 20,
                },
            }
        ],
    },
    "meta": {
        "name": "[name]",
        "version": "1.0",
    },
}

os.makedirs("/content/ai-toolkit/config", exist_ok=True)
CONFIG_PATH = "/content/ai-toolkit/config/style_lora.yaml"
with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

print(f"✅ 训练配置已写入: {CONFIG_PATH}")

In [ ]:
# @title 5. 启动 LoRA 训练、归档到 models/loras/ 并刷盘同步 Drive
import glob
import os
import shutil
import subprocess
from google.colab import drive

RUN_NAME = globals().get("RUN_NAME", "custom_style_v1")
CONFIG_PATH = "/content/ai-toolkit/config/style_lora.yaml"
STUDIO_DIR = "/content/drive/MyDrive/GoogleOneAI_Studio"
RUN_OUT_DIR = f"{STUDIO_DIR}/training_runs/{RUN_NAME}"
LORA_DIR = f"{STUDIO_DIR}/models/loras"
os.makedirs(LORA_DIR, exist_ok=True)

proc = subprocess.run(
    ["python3", "/content/ai-toolkit/run.py", CONFIG_PATH],
    cwd="/content/ai-toolkit",
    check=False,
)
if proc.returncode != 0:
    raise RuntimeError(f"❌ 训练异常退出 (code={proc.returncode})，请检查上方输出日志。")

final_lora = os.path.join(RUN_OUT_DIR, f"{RUN_NAME}.safetensors")
if not os.path.exists(final_lora):
    candidates = sorted(
        glob.glob(os.path.join(RUN_OUT_DIR, "*.safetensors")),
        key=os.path.getmtime,
    )
    if not candidates:
        raise FileNotFoundError(f"❌ 未在 {RUN_OUT_DIR} 找到输出的 .safetensors 权重。")
    final_lora = candidates[-1]

dst_lora = os.path.join(LORA_DIR, f"{RUN_NAME}.safetensors")
shutil.copy2(final_lora, dst_lora)
size_mb = os.path.getsize(dst_lora) / (1024**2)

print(f"🎉 LoRA 训练完成并归档: {dst_lora} ({size_mb:.1f} MB)")
print(f"🖼️ 过程采样对比图保存在: {RUN_OUT_DIR}/samples/")
print("⏳ 正在刷盘写入 Google Drive...")
drive.flush_and_unmount()
print("✅ 权重已永久同步到云盘！")
print("👉 下一步：打开 01_comfyui_studio.ipynb，在工作流中选中 LoraLoader 按 Ctrl+B 解除 Bypass 即可使用。")
print("⚠️ 请立即点击：Runtime -> Disconnect and delete runtime 释放 GPU！")